Model Training and Evaluation: Adversarial Phishing DetectionThis notebook is dedicated to training and evaluating the final Machine Learning (ML) and Deep Learning (DL) models for the phishing detection system. We separate this step from feature engineering to maintain a clean, modular, and reproducible workflow.1. Data Preparation and SplittingThe primary goal of this step is to load the dataset containing all the engineered features (created in the project_overview.ipynb and feature_engineer.py) and split it into training and testing sets.The input data is assumed to be stored as ../processed/sessions_engineered.csv.1.1 Loading DataWe load the data, define the features X and the target label X, and then perform an 80/20 train-test split, ensuring the split is stratified to maintain the original class distribution in both sets.

MLP Deep learning model

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import joblib
import random

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

df = pd.read_csv("../../processed/Feature.csv")
df = df.dropna()

label_candidates = ['label', 'Label', 'target', 'Target', 'y']
label_col = next((c for c in label_candidates if c in df.columns), df.columns[-1])

X = df.drop(columns=[label_col])
y = df[label_col]

if y.dtype == object or not np.issubdtype(y.dtype, np.number):
    le = LabelEncoder()
    y = le.fit_transform(y)

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
])

X_processed = preprocessor.fit_transform(X)

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accs, pres, recs, f1s = [], [], [], []

for train_idx, test_idx in kf.split(X_processed, y):
    X_train, X_test = X_processed[train_idx], X_processed[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    classes = np.unique(y_train)
    cw_vals = compute_class_weight('balanced', classes=classes, y=y_train)
    class_weights = dict(zip(classes, cw_vals))

    model = Sequential([
        Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation='relu', kernel_regularizer=l2(0.001)),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])

    es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    model.fit(X_train, y_train,
              validation_split=0.2,
              epochs=50,
              batch_size=16,
              verbose=0,
              callbacks=[es],
              class_weight=class_weights)

    y_pred = (model.predict(X_test) > 0.5).astype(int)

    accs.append(accuracy_score(y_test, y_pred))
    pres.append(precision_score(y_test, y_pred))
    recs.append(recall_score(y_test, y_pred))
    f1s.append(f1_score(y_test, y_pred))

print("Accuracy:", np.mean(accs))
print("Precision:", np.mean(pres))
print("Recall:", np.mean(recs))
print("F1:", np.mean(f1s))

model.save("mlp_stable_model.keras")
joblib.dump(preprocessor, "mlp_preprocessor.pkl")

test_df = pd.read_csv("../../processed/test_mlp.csv")

if 'label' in test_df.columns:
    y_test = test_df['label']
    X_test = test_df.drop(columns=['label'])
else:
    y_test = None
    X_test = test_df.copy()

train_cols = X.columns.tolist()

for c in train_cols:
    if c not in X_test.columns:
        X_test[c] = 0 if c in numeric_cols else ""

extra = [c for c in X_test.columns if c not in train_cols]
if extra:
    X_test = X_test.drop(columns=extra)

X_test = X_test[train_cols]
X_test_processed = preprocessor.transform(X_test)

y_pred = (model.predict(X_test_processed) > 0.5).astype(int)

if y_test is not None:
    print("Test Accuracy:", accuracy_score(y_test, y_pred))
    print("Test Precision:", precision_score(y_test, y_pred))
    print("Test Recall:", recall_score(y_test, y_pred))
    print("Test F1:", f1_score(y_test, y_pred))
else:
    print(y_pred)


c:\Python313\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 178ms/step
Accuracy: 0.8
Precision: 0.9
Recall: 0.9
F1: 0.8666666666666666
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 180ms/step
Test Accuracy: 0.5
Test Precision: 0.5
Test Recall: 1.0
Test F1: 0.6666666666666666


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib

train_path = "../../processed/Feature.csv"
test_path  = "../../processed/test_mlp.csv"

df_train = pd.read_csv(train_path)
df_test  = pd.read_csv(test_path)

df = pd.concat([df_train, df_test], ignore_index=True)

df['url'] = df['url'].fillna('').astype(str)
df['url_length'] = df['url'].str.len()
df['num_dots'] = df['url'].str.count(r'\.')
df['has_https'] = df['url'].str.startswith('https').astype(int)
df['num_digits'] = df['url'].str.count(r'\d')
df['num_special_chars'] = df['url'].str.count(r'[^A-Za-z0-9]').astype(int)
df['has_ip'] = df['url'].apply(lambda x: 1 if any(part.isdigit() for part in x.split('.')) else 0)
df['url_entropy'] = df['url'].apply(lambda x: len(set(x)) / len(x) if len(x) > 0 else 0)

le_location = LabelEncoder()
df["Location_encoded"] = le_location.fit_transform(df["Location"].astype(str))

feature_columns = [
    'TransactionAmount', 'CustomerAge', 'AccountBalance',
    'url_length', 'num_dots', 'has_https', 'num_digits',
    'num_special_chars', 'has_ip', 'url_entropy', 'Location_encoded'
]

X = df[feature_columns].fillna(0)
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=30,
    max_depth=3,
    min_samples_split=20,
    min_samples_leaf=10,
    max_features=3,
    bootstrap=True,
    max_samples=0.7,
    oob_score=True,
    criterion="entropy",
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
train_accuracy = accuracy_score(y_train, y_pred_train)
train_precision = precision_score(y_train, y_pred_train)
train_recall = recall_score(y_train, y_pred_train)
train_f1 = f1_score(y_train, y_pred_train)

y_pred_test = model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred_test)
test_precision = precision_score(y_test, y_pred_test)
test_recall = recall_score(y_test, y_pred_test)
test_f1 = f1_score(y_test, y_pred_test)

print("\nTRAIN PERFORMANCE")
print("="*50)
print(f"Train Accuracy:  {train_accuracy:.4f}")
print(f"Train Precision: {train_precision:.4f}")
print(f"Train Recall:    {train_recall:.4f}")
print(f"Train F1:        {train_f1:.4f}")

print("\nTEST PERFORMANCE")
print("="*50)
print(f"Test Accuracy:   {test_accuracy:.4f}")
print(f"Test Precision:  {test_precision:.4f}")
print(f"Test Recall:     {test_recall:.4f}")
print(f"Test F1:         {test_f1:.4f}")

joblib.dump(model, "random_forest_final.pkl")
print("\nCompleted Successfully!")



TRAIN PERFORMANCE
Train Accuracy:  0.7143
Train Precision: 0.7143
Train Recall:    1.0000
Train F1:        0.8333

TEST PERFORMANCE
Test Accuracy:   0.6000
Test Precision:  0.6000
Test Recall:     1.0000
Test F1:         0.7500

Feature Importance Ranking:
1. Location_encoded: 0.0000
2. url_entropy: 0.0000
3. has_ip: 0.0000
4. num_special_chars: 0.0000
5. num_digits: 0.0000
6. has_https: 0.0000
7. num_dots: 0.0000
8. url_length: 0.0000
9. AccountBalance: 0.0000
10. CustomerAge: 0.0000
11. TransactionAmount: 0.0000

Completed Successfully!
